In [2]:
!pip install langchain-google-genai

  Using cached langchain_google_genai-2.1.7-py3-none-any.whl.metadata (7.0 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached google_ai_generativelanguage-0.6.18-py3-none-any.whl.metadata (9.8 kB)
  Using cached google_api_core-2.25.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached proto_plus-1.26.1-py3-none-any.whl.metadata (2.2 kB)
  Using cached grpcio_status-1.73.1-py3-none-any.whl.metadata (1.1 kB)
  Using cached protobuf-6.31.1-cp310-abi3-win_amd64.whl.metadata (593 bytes)
Using cached langchain_google_genai-2.1.7-py3-none-any.whl (47 kB)
Using cached filetype-1.2.0-py2.py3-none-any.whl (19 kB)
Using cached google_ai_generativelanguage-0.6.18-py3-none-any.whl (1.4 MB)
Using cached google_api_core-2.25.1-py3-none-any.whl (160 kB)
Using cached proto_plus-1.26.1-py3-none-any.whl (50 kB)
Using cached grpcio_status-1.73.1-py3-none-any.whl (14 kB)
Using cached protobuf-6.31.1-cp310-abi3-win_amd64.whl (435 kB)
  Attempting uninstall: protobuf
    F

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.44.1 requires protobuf<6,>=3.20, but you have protobuf 6.31.1 which is incompatible.


In [21]:
import pandas as pd
from langchain.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA
from langchain_core.documents import Document
import os
from dotenv import load_dotenv

In [4]:
base = pd.read_csv('historico_OS_manut_23_25.csv')

In [5]:
df = base[['DEFEITO', 'SOLUCAO', 'SERVICO_REALIZADO', 'ATENDIMENTO_REMOTO', 'TECNOLOGIA', 'EQUIPAMENTO', 'CAUSA']]
df = df.dropna(subset=['SERVICO_REALIZADO'])
df

,DEFEITO,SOLUCAO,SERVICO_REALIZADO,ATENDIMENTO_REMOTO,TECNOLOGIA,EQUIPAMENTO,CAUSA
0,-,CHAMADO EM BRANCO;,Requisição de Serviço,Sim,Voice Manager,Broadworks,NaN
2,Solicito o recolhimento de dois aparelh...,30/08/2024 - Simone Pimenta - Tec. Eduardo cie...,Requisição de Serviço,Não,Voz Tradicional,Aparelho IP,Defeito no hardware
3,Cliente: André ou Fernando\r\n Circ...,Enviado email para analistas algar:\r\nRealiza...,Requisição de Serviço,Sim,Voice Manager,Broadworks,Configuração
4,OS SOM: 0016357605\r\n Cliente: Mat...,Realizado novos testes e apos normalização de ...,Manutenção Corretiva - Inoperância,Sim,Voice Manager,SBC,Capacidade
5,Circuito: 0000253495\r\n Protocolo: ...,chamado encerrado:\r\nFalha na certificação do...,Manutenção Corretiva,Sim,Voice Manager,Servidor,Atualização de software
...,...,...,...,...,...,...,...
81914,Yealink T43U\r\nDDR Tipo 2 \r\nMAC: 24:9A:D8...,Associado o dispositivo Yealink ao usuário sol...,Manutenção Corretiva,Sim,Voice Manager,Aparelho IP,Configuração
81915,Yealink T43U\r\nDDR Tipo 2 \r\nMAC: 24:9A:D8...,Associado o dispositivo Yealink ao usuário sol...,Manutenção Corretiva,Sim,Voice Manager,Aparelho IP,Configuração
81916,Yealink T43U\r\nDDR\r\nTipo 2\r\nMAC : 24:9A:D...,"Dispositivo associado ao ramal 1567, efetuado ...",Manutenção Corretiva,Sim,Voice Manager,Broadworks,Configuração
81917,YUTAKA DO BRASIL LTDA.\r\nCPF/CNPJ\r\n04841302...,CAUSA RAÍZ : CLIENTE\r\nIMPACTO : CONFIGURAÇÃO...,Manutenção Corretiva,Sim,Voice Manager,Softphone,Configuração


In [9]:
documents = []
for index, row in df.iterrows():
    page_content = f"Defeito: {row['DEFEITO']}. Solução: {row['SOLUCAO']}"
    metadata = {
        "SERVICO_REALIZADO": row['SERVICO_REALIZADO'],
        "ATENDIMENTO_REMOTO": row['ATENDIMENTO_REMOTO'],
        "TECNOLOGIA": row['TECNOLOGIA'],
        "EQUIPAMENTO": row['EQUIPAMENTO'],
        "CAUSA": row['CAUSA']
    }
    documents.append(Document(page_content=page_content, metadata=metadata))
    
documents

[Document(metadata={'SERVICO_REALIZADO': 'Requisição de Serviço', 'ATENDIMENTO_REMOTO': 'Sim', 'TECNOLOGIA': 'Voice Manager', 'EQUIPAMENTO': 'Broadworks', 'CAUSA': nan}, page_content='Defeito: -. Solução: CHAMADO EM BRANCO;'),
 Document(metadata={'SERVICO_REALIZADO': 'Requisição de Serviço', 'ATENDIMENTO_REMOTO': 'Não', 'TECNOLOGIA': 'Voz Tradicional', 'EQUIPAMENTO': 'Aparelho IP', 'CAUSA': 'Defeito no hardware'}, page_content='Defeito:        Solicito o recolhimento de dois aparelhos YNIFY IP35 que estão com a trava RJ 45 quebrada e falha no software relatados  na ordem de serviço 495469, pois a método já mandou  dois aparelhos novos para substitui-los.\r\nAtt, Ricardo 4299 \r\n\n        OS aberta por: ricardo.rocha@tst.jus.br\n        contato 1:  61984280405\n        contato 2:  \n        . Solução: 30/08/2024 - Simone Pimenta - Tec. Eduardo ciente , vai no TST no dia 02/09 e vai recolher os aparelhos.\r\n\r\n\r\nO texto abaixo é de responsabilidade de VITORIO:\r\n\r\n"Cliente solici

In [12]:
# Carrega as variáveis de ambiente do arquivo .env
print("Carregando variáveis de ambiente do arquivo .env...")
load_dotenv()

Carregando variáveis de ambiente do arquivo .env...


True

In [15]:
# Inicializar o modelo de embeddings do Google
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

# Define o diretório onde o banco de dados será salvo
persist_directory = 'chroma_db_gemini'

In [16]:
# Cria o banco de dados vetorial a partir dos documentos
vectordb = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=persist_directory
)

In [ ]:
# Carrega a Chave da API
print("Carregando chave de API e modelo de embeddings...")
load_dotenv()

Carregando chave de API e modelo de embeddings...


True

In [18]:
# Carrega
persist_directory = 'chroma_db_gemini'
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embeddings)

print("Banco de dados vetorial carregado com sucesso!")

Banco de dados vetorial carregado com sucesso!


C:\Users\mathe\AppData\Local\Temp\ipykernel_21748\234213599.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectordb = Chroma(persist_directory=persist_directory, embedding_function=embeddings)


In [19]:
# Inicializa o LLM que irá gerar as respostas.
llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro-latest", temperature=0.0)

print("Modelo de Linguagem (LLM) inicializado com sucesso!")

Modelo de Linguagem (LLM) inicializado com sucesso!


In [20]:
# Cria o retriever a partir do seu banco de dados vetorial.
retriever = vectordb.as_retriever(search_kwargs={'k': 3})

print("Retriever criado. Pronto para buscar documentos.")

Retriever criado. Pronto para buscar documentos.


In [22]:
# Cria a cadeia que une o LLM e o Retriever.
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

print("Cadeia de Pergunta e Resposta (RAG) pronta!")

Cadeia de Pergunta e Resposta (RAG) pronta!


In [23]:
# Pergunta
pergunta = "Como foi resolvido o problema de lentidão no sistema?"

# Invoca a cadeia com sua pergunta
response = qa_chain.invoke(pergunta)

# Imprime a resposta
print("\n--- Resposta Gerada ---")
print(response['result'])

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateContentInputTokensPerModelPerMinute-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-1.5-pro"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
}
violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-1.5-pro"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
}
violations {
  quota_metric: "generativelanguage.googleapis.com/gener

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateContentInputTokensPerModelPerMinute-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-1.5-pro"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
}
violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-1.5-pro"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
}
violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions 